# INX Future Inc. — Employee Performance Analysis
## Complete End-to-End Data Science Project

**IABAC Project 2 | Project Code: 10281**

**Objective:** Analyze employee data to understand patterns associated with performance ratings, compare departments, explore important performance-related factors, and build machine-learning models to predict `PerformanceRating`.

### Notebook workflow
1. Business understanding and project objectives
2. Dataset loading and data-quality checks
3. Data preprocessing
4. Exploratory Data Analysis (EDA) and visualizations
5. Department-wise performance comparison
6. Machine-learning preparation and model building
7. Model evaluation and comparison
8. Feature importance and example prediction
9. Findings, recommendations, limitations, and conclusion

### Before running
- Keep `INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1.8.xls` in the same folder as this notebook, or update `file_path` in the loading cell.
- If the Excel reader is missing, install it using `%pip install xlrd`, restart the kernel, and run the notebook from the beginning.
- Run cells from top to bottom.
- This notebook uses the original Excel dataset and reuses the same DataFrame throughout. It does not depend on a separately saved cleaned CSV file.

**Important:** Descriptive analysis and model feature importance show associations or predictive patterns. They do not prove that a factor causes employee performance outcomes.


## 1. Business understanding

INX Future Inc. has observed concerns related to employee performance and wants data-driven insights to support management decisions.

### Project questions
- How is employee performance distributed?
- How do performance ratings differ across departments?
- Which employee and workplace factors show differences across performance-rating groups?
- Can a machine-learning model predict the performance rating?
- What practical recommendations can be made from the analysis?

### Expected deliverables
- Department-wise performance analysis
- Evidence-based review of important performance-related factors
- A trained model to predict employee performance rating
- Recommendations for management


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 2. Load the Excel file
If pandas says an Excel engine is missing, install `xlrd` with `pip install xlrd`, restart the kernel, and rerun.

In [ ]:
file_path = "INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1.8.xls"
df = pd.read_excel(file_path)

print("Shape:", df.shape)
df.head()

## 3. Understand rows, columns, and data types

In [ ]:
print("Rows and columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
df.info()

## 4. Check missing values
Count missing cells and their percentage. Do not fill them automatically; first understand which columns are affected.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)

missing_summary

## 5. Check and remove exact duplicate rows
We remove only fully duplicated rows. We do not remove employees merely because some values look similar.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Shape after duplicate check:", df.shape)

## 6. Clean column names and surrounding spaces in text
This removes accidental spaces at the beginning or end of labels. It does not rename categories or change their meaning.

In [ ]:
df.columns = df.columns.str.strip()

text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Text and column-name spaces checked.")

## 7. Check employee identifier
`EmpNumber` is useful for identifying records. Keep it in the cleaned file, but normally exclude it from model input because it is an identifier, not an employee-performance measurement.

In [ ]:
if "EmpNumber" in df.columns:
    print("Unique employee numbers:", df["EmpNumber"].nunique())
    print("Repeated employee numbers:", df["EmpNumber"].duplicated().sum())
else:
    print("EmpNumber column not found; check column names.")

## 8. Inspect the target: PerformanceRating
The project target is the employee performance rating. Check the available values and class distribution. Do not change ratings without evidence from the project documentation.

In [ ]:
if "PerformanceRating" in df.columns:
    print("Rating values:", sorted(df["PerformanceRating"].dropna().unique()))
    print(df["PerformanceRating"].value_counts().sort_index())
    print((df["PerformanceRating"].value_counts(normalize=True).sort_index() * 100).round(2))

## 9. Numerical summary
Minimum, maximum, mean, and quartiles help flag unusual values. A value outside an expected range should be investigated against the source, not automatically deleted.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns
df[numeric_columns].describe().T

## 10. Inspect rating-scale and categorical values
Review unique values for satisfaction, involvement, education, and performance scales. Also inspect text categories such as department, job role, travel frequency, overtime, gender, and marital status.

In [ ]:
scale_columns = [
    "EmpEducationLevel", "EmpEnvironmentSatisfaction",
    "EmpJobInvolvement", "EmpJobLevel", "EmpJobSatisfaction",
    "EmpRelationshipSatisfaction", "EmpWorkLifeBalance",
    "PerformanceRating"
]
for col in scale_columns:
    if col in df.columns:
        print(f"{col}: {sorted(df[col].dropna().unique())}")

for col in df.select_dtypes(include="object").columns:
    print(f"\\n{col} ({df[col].nunique()} unique values)")
    print(df[col].value_counts(dropna=False).head(25))

## 11. Check numeric ranges and experience values
These checks help identify negative experience or unexpected ranges. They only report potential issues; do not delete records without checking the source.

In [ ]:
check_columns = [
    "Age", "DistanceFromHome", "EmpHourlyRate",
    "EmpLastSalaryHikePercent", "NumCompaniesWorked",
    "TotalWorkExperienceInYears", "TrainingTimesLastYear",
    "ExperienceYearsAtThisCompany", "ExperienceYearsInCurrentRole",
    "YearsSinceLastPromotion", "YearsWithCurrManager"
]
for col in check_columns:
    if col in df.columns:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}, missing={df[col].isna().sum()}")

experience_columns = [
    "TotalWorkExperienceInYears", "ExperienceYearsAtThisCompany",
    "ExperienceYearsInCurrentRole", "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]
for col in experience_columns:
    if col in df.columns:
        print(col, "negative values:", (df[col] < 0).sum())

## 12. Final data-quality summary
Recheck missing values and duplicates after the review. If there are unresolved unusual values, document them before using this dataset for modelling.

In [ ]:
print("Final shape:", df.shape)
print("Missing cells:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

quality_summary = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum(),
    "Unique_Values": df.nunique()
})
quality_summary

## 4. Continue with the reviewed DataFrame

The reviewed DataFrame `df` remains in memory. We will use it directly for EDA and model building, so a separate cleaned CSV file is not required.

The original Excel file is not overwritten.


# Part 2 — Exploratory Data Analysis (EDA)

EDA helps us understand the structure of the dataset, identify patterns, compare employee groups, and generate questions for further investigation.

The following analysis uses the same `df` DataFrame prepared in Part 1.


## 1. Import visualization libraries and continue with the dataset

We already loaded and reviewed the original Excel dataset in Part 1. Here we import the visualization libraries and continue using the existing `df`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Dataset shape:", df.shape)
df.head()


## 2. Understand the data
Review data types and descriptive statistics before plotting.

In [ ]:
df.info()
df.describe(include="all").T

## 3. Performance rating distribution
First check how many employees received each rating and what percentage they represent. This also helps identify class imbalance for future classification.

In [ ]:
rating_counts = df["PerformanceRating"].value_counts().sort_index()
rating_percent = (df["PerformanceRating"].value_counts(normalize=True).sort_index() * 100)

pd.DataFrame({
    "Employee_Count": rating_counts,
    "Percentage": rating_percent.round(2)
})

In [ ]:
rating_counts.plot(kind="bar", figsize=(7,5))
plt.title("Distribution of Employee Performance Ratings")
plt.xlabel("Performance Rating")
plt.ylabel("Number of Employees")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretation prompt:** Which rating is most common? Are the rating classes balanced? Mention the actual counts and percentages from your output.

## 4. Department-wise performance
Compare employee count, average performance rating, and the percentage of employees with rating 2. The average and low-rating percentage answer different questions.

In [ ]:
department_summary = df.groupby("EmpDepartment").agg(
    Employee_Count=("PerformanceRating", "size"),
    Average_Performance=("PerformanceRating", "mean"),
    Rating_2_Count=("PerformanceRating", lambda x: (x == 2).sum()),
    Rating_2_Percentage=("PerformanceRating", lambda x: (x == 2).mean() * 100)
).sort_values("Average_Performance")

department_summary.round(2)

In [ ]:
department_summary["Average_Performance"].plot(kind="bar", figsize=(9,5))
plt.title("Average Performance Rating by Department")
plt.xlabel("Department")
plt.ylabel("Average Performance Rating")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
department_summary["Rating_2_Percentage"].sort_values().plot(kind="bar", figsize=(9,5))
plt.title("Percentage of Rating 2 Employees by Department")
plt.xlabel("Department")
plt.ylabel("Rating 2 Employees (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11,6))
sns.countplot(data=df, x="EmpDepartment", hue="PerformanceRating")
plt.title("Performance Rating Counts Across Departments")
plt.xlabel("Department")
plt.ylabel("Employee Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Interpretation prompt:** Which department has the lowest average rating? Which has the highest percentage of rating 2? Do not confuse the two measures.

## 5. Satisfaction and involvement comparisons
Compare environment satisfaction, job satisfaction, job involvement, relationship satisfaction, and work-life balance across performance ratings. These are rating-scale variables; tiny differences in averages should not be exaggerated.

In [ ]:
satisfaction_columns = [
    "EmpEnvironmentSatisfaction", "EmpJobSatisfaction",
    "EmpJobInvolvement", "EmpRelationshipSatisfaction",
    "EmpWorkLifeBalance"
]
available_satisfaction = [c for c in satisfaction_columns if c in df.columns]

satisfaction_means = df.groupby("PerformanceRating")[available_satisfaction].mean().T
satisfaction_means.round(2)

In [ ]:
satisfaction_means.plot(kind="bar", figsize=(11,6))
plt.title("Average Satisfaction and Involvement by Performance Rating")
plt.xlabel("Employee Factor")
plt.ylabel("Average Score")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Performance Rating")
plt.tight_layout()
plt.show()

In [ ]:
if "EmpEnvironmentSatisfaction" in df.columns:
    print(df.groupby("PerformanceRating")["EmpEnvironmentSatisfaction"].mean().round(2))
    sns.boxplot(data=df, x="PerformanceRating", y="EmpEnvironmentSatisfaction")
    plt.title("Environment Satisfaction Distribution by Performance Rating")
    plt.xlabel("Performance Rating")
    plt.ylabel("Environment Satisfaction")
    plt.tight_layout()
    plt.show()

**Earlier observed pattern to recheck:** environment satisfaction averages were approximately 1.58 for rating 2, 2.91 for rating 3, and 3.08 for rating 4. This is a strong descriptive difference worth investigating. It is an association, not proof of cause.

## 6. Department-wise work-life balance comparison
Compare average performance and average work-life balance within each department. A department-level comparison does not tell us what happens to every individual employee.

In [ ]:
department_comparison = df.groupby("EmpDepartment")[
    ["PerformanceRating", "EmpWorkLifeBalance"]
].mean().sort_values("PerformanceRating")

department_comparison.round(2)

In [ ]:
department_comparison.plot(kind="bar", figsize=(10,6))
plt.title("Department-wise Performance and Work-Life Balance")
plt.xlabel("Department")
plt.ylabel("Average Score")
plt.xticks(rotation=45, ha="right")
plt.legend(["Performance Rating", "Work-Life Balance"])
plt.tight_layout()
plt.show()

**Interpretation prompt:** Do departments with lower average work-life balance always have lower performance ratings? Use the actual department comparison to answer.

## 7. Overtime vs. performance
Use row percentages. Each overtime group adds to approximately 100%, so we can compare the rating distribution among employees who work overtime and those who do not.

In [ ]:
overtime_table = pd.crosstab(
    df["OverTime"], df["PerformanceRating"], normalize="index"
).mul(100).round(2)
overtime_table

In [ ]:
overtime_table.plot(kind="bar", figsize=(8,5))
plt.title("Performance Rating Distribution by Overtime Status")
plt.xlabel("Overtime")
plt.ylabel("Percentage within Overtime Group")
plt.xticks(rotation=0)
plt.legend(title="Performance Rating")
plt.tight_layout()
plt.show()

**Earlier observed result to recheck:** rating 2 was about 18.30% among employees with no overtime and 11.05% among employees with overtime. This does not support the claim that overtime is associated with a higher percentage of rating 2 in this dataset. It does not prove overtime improves performance.

## 8. Training frequency
Compare average training times and inspect the distribution. The number of sessions does not measure training quality or relevance.

In [ ]:
print(df.groupby("PerformanceRating")["TrainingTimesLastYear"].mean().round(2))
sns.boxplot(data=df, x="PerformanceRating", y="TrainingTimesLastYear")
plt.title("Training Times Last Year by Performance Rating")
plt.xlabel("Performance Rating")
plt.ylabel("Training Times Last Year")
plt.tight_layout()
plt.show()

## 9. Experience and career progression
Compare total work experience, time at the current company, time in the current role, time since last promotion, and time with the current manager. These variables overlap, so do not treat them as five independent causes.

In [ ]:
experience_columns = [
    "TotalWorkExperienceInYears",
    "ExperienceYearsAtThisCompany",
    "ExperienceYearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]
available_experience = [c for c in experience_columns if c in df.columns]
experience_means = df.groupby("PerformanceRating")[available_experience].mean().T
experience_means.round(2)

In [ ]:
experience_means.plot(kind="bar", figsize=(12,6))
plt.title("Experience and Career History by Performance Rating")
plt.xlabel("Experience Measure")
plt.ylabel("Average Years")
plt.xticks(rotation=40, ha="right")
plt.legend(title="Performance Rating")
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(data=df, x="PerformanceRating", y="YearsSinceLastPromotion")
plt.title("Years Since Last Promotion by Performance Rating")
plt.xlabel("Performance Rating")
plt.ylabel("Years Since Last Promotion")
plt.tight_layout()
plt.show()

**Earlier observed pattern to recheck:** rating-2 employees had higher averages for total work experience (12.75 years), current-company experience (9.10 years), current-role experience (5.79 years), and years since last promotion (3.70 years). This describes group differences; it does not mean experienced employees perform worse or that delayed promotion causes lower ratings.

## 10. Salary and compensation
Compare hourly rate and last salary-hike percentage. A salary hike could be a reward for previous performance, so this dataset alone cannot establish whether the hike came before or after the rating.

In [ ]:
compensation_columns = ["EmpHourlyRate", "EmpLastSalaryHikePercent"]
available_compensation = [c for c in compensation_columns if c in df.columns]
compensation_means = df.groupby("PerformanceRating")[available_compensation].mean().T
compensation_means.round(2)

In [ ]:
compensation_means.plot(kind="bar", figsize=(9,5))
plt.title("Average Compensation Measures by Performance Rating")
plt.xlabel("Compensation Measure")
plt.ylabel("Average Value")
plt.xticks(rotation=0)
plt.legend(title="Performance Rating")
plt.tight_layout()
plt.show()

**Earlier observed salary-hike averages to recheck:** rating 2 = 15.07%, rating 3 = 14.43%, rating 4 = 20.70%. The rating-2 average is slightly above rating 3, while rating 4 is substantially higher. This is not a simple step-by-step increase and does not show that salary hikes cause better performance.

## 11. Age comparison
Compare age distributions and averages. A small difference between averages should not be treated as an important predictor without further testing.

In [ ]:
print(df.groupby("PerformanceRating")["Age"].mean().round(2))
sns.boxplot(data=df, x="PerformanceRating", y="Age")
plt.title("Age Distribution by Performance Rating")
plt.xlabel("Performance Rating")
plt.ylabel("Age")
plt.tight_layout()
plt.show()

## 12. Business travel frequency
This row-normalized table compares the percentage of each travel-frequency group receiving each performance rating.

In [ ]:
travel_table = pd.crosstab(
    df["BusinessTravelFrequency"],
    df["PerformanceRating"],
    normalize="index"
).mul(100).round(2)
travel_table

In [ ]:
travel_table.plot(kind="bar", figsize=(8,5))
plt.title("Performance Rating Distribution by Business Travel Frequency")
plt.xlabel("Business Travel Frequency")
plt.ylabel("Percentage within Travel Group")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Performance Rating")
plt.tight_layout()
plt.show()

## 13. Finance department deep dive
Finance had the lowest department-average performance rating in the earlier analysis. Here we reproduce a focused comparison. Be cautious because this department has fewer employees than some other departments.

In [ ]:
finance_df = df[df["EmpDepartment"] == "Finance"].copy()
print("Finance employee count:", len(finance_df))
print(finance_df["PerformanceRating"].value_counts().sort_index())

finance_columns = [
    "EmpEnvironmentSatisfaction", "EmpJobSatisfaction",
    "EmpJobInvolvement", "EmpRelationshipSatisfaction",
    "EmpWorkLifeBalance", "TrainingTimesLastYear",
    "YearsSinceLastPromotion", "ExperienceYearsAtThisCompany"
]
available_finance_columns = [c for c in finance_columns if c in finance_df.columns]
finance_df.groupby("PerformanceRating")[available_finance_columns].mean().T.round(2)

In [ ]:
sns.countplot(data=finance_df, x="PerformanceRating")
plt.title("Finance Department: Performance Rating Counts")
plt.xlabel("Performance Rating")
plt.ylabel("Employee Count")
plt.tight_layout()
plt.show()

## 14. Correlation heatmap
Correlation summarizes how numerical variables move together. It does not prove causation. Some scale variables are ordinal, so interpret Pearson correlations carefully.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(15,11))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numerical Variables")
plt.tight_layout()
plt.show()

## 15. Summary table of candidate factors
This table collects group means for selected variables. It is not a ranking of feature importance.

In [ ]:
selected_factors = [
    "EmpEnvironmentSatisfaction", "EmpJobInvolvement",
    "EmpJobSatisfaction", "EmpRelationshipSatisfaction",
    "EmpWorkLifeBalance", "EmpLastSalaryHikePercent",
    "TrainingTimesLastYear", "TotalWorkExperienceInYears",
    "ExperienceYearsAtThisCompany", "ExperienceYearsInCurrentRole",
    "YearsSinceLastPromotion", "EmpHourlyRate", "Age"
]
available_factors = [c for c in selected_factors if c in df.columns]
df.groupby("PerformanceRating")[available_factors].mean().round(2).T

## 16. EDA conclusions — complete from your outputs

Write evidence-based answers:

1. Which performance rating is most common?
2. Which department has the lowest average performance rating?
3. Which department has the highest percentage of rating 2 employees?
4. Which satisfaction variable shows the clearest difference across rating groups?
5. How does time since last promotion differ across ratings?
6. What patterns appear in employee experience and tenure?
7. What do overtime and training comparisons show?
8. What limitations should be mentioned?

### Current exploratory clues
- Environment satisfaction showed a large difference between rating 2 and ratings 3/4.
- Years since last promotion and the experience measures were higher on average in the rating-2 group.
- Job involvement, job satisfaction, relationship satisfaction, and training frequency had fairly similar group averages.
- Overtime did not show a higher percentage of rating 2 among overtime employees in the earlier comparison.
- Salary hike averages were highest for rating 4, but rating 2 was slightly higher than rating 3.

These are descriptive observations. They do not prove causes or establish the final top three predictive factors.

## 17. Next stage after EDA

1. Confirm the target variable and project objective.
2. Use suitable statistical tests to examine group differences.
3. Encode categorical variables appropriately.
4. Split training and test data before fitting data transformations.
5. Train baseline classification models.
6. Evaluate with a confusion matrix, precision, recall, F1-score, and class-aware metrics.
7. Use model-based feature importance or explainability methods to identify predictive factors.

Exclude `EmpNumber` from model inputs. Check for data leakage, and do not describe model feature importance as proof of causation.

# Part 3 — Machine Learning Model Building

In this section, we prepare the data for classification and train models to predict `PerformanceRating`.

We will compare Logistic Regression and Random Forest using a train/test split and evaluation metrics. The preprocessing steps are placed inside Scikit-learn pipelines to help prevent data leakage.


## 1. Import the libraries
We use pandas to load the Excel data, scikit-learn to prepare the data and train models, and matplotlib/seaborn to display evaluation results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Confirm the dataset for model building

We will use the same original-data DataFrame `df` loaded and checked in Part 1. No cleaned CSV file is loaded.


In [ ]:
# Reuse the DataFrame prepared in Part 1
print("Dataset shape:", df.shape)
df.head()


## 3. Basic data checks
Check the columns, missing values, duplicate rows, and target distribution before modelling.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\\nMissing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\\nPerformance rating counts:")
print(df["PerformanceRating"].value_counts().sort_index())

## 4. Separate input features (X) and target (y)

- **X** contains the information used by the model to make a prediction.
- **y** contains the answer the model learns to predict: `PerformanceRating`.

We remove `EmpNumber` because it is an identifier, not a meaningful employee characteristic. We also remove the target from X so the model cannot see the answer while learning.

In [ ]:
target = "PerformanceRating"

X = df.drop(columns=[target, "EmpNumber"], errors="ignore")
y = df[target]

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", sorted(y.unique()))

## 5. Identify numerical and categorical columns

Machine-learning models need numerical input. Numerical columns can be scaled, while text categories such as department and job role need to be converted into indicator columns using One-Hot Encoding.

We identify the column types automatically so the code can adapt to the dataset.

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical columns:", numeric_features)
print("\\nCategorical columns:", categorical_features)

## 6. Split the data into training and testing sets

- Training data teaches the model.
- Testing data checks how well the model predicts records it did not train on.

We use 80% for training and 20% for testing. `stratify=y` helps preserve the proportions of ratings 2, 3, and 4 in both sets. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

print("\\nTraining target percentages:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\\nTesting target percentages:")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))

## 7. Prepare numerical and categorical data

The preprocessing steps are placed inside a pipeline. This is important because the imputer, scaler, and encoder are fitted using training data only, then applied to the test data.

- Numerical columns: fill any missing values with the median, then standardize the values.
- Categorical columns: fill any missing values with the most frequent category, then apply One-Hot Encoding.
- `handle_unknown="ignore"` allows the model to handle a category in test data that was not present in training data.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessing pipeline created.")

## 8. Build Model 1: Logistic Regression

Logistic Regression is a classification algorithm. It learns patterns in the input columns and estimates the probability of each performance-rating class.

We put preprocessing and the model together in one pipeline so that the same preparation is applied consistently.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression model training completed.")

## 9. Evaluate Model 1
We check accuracy, the classification report, and the confusion matrix. Precision, recall, and F1-score help us understand performance for each rating, not just overall accuracy.

In [ ]:
logistic_predictions = logistic_model.predict(X_test)

print("Logistic Regression Accuracy:",
      round(accuracy_score(y_test, logistic_predictions), 4))

print("\\nClassification Report:")
print(classification_report(y_test, logistic_predictions, zero_division=0))

In [ ]:
labels = sorted(y.unique())
cm_logistic = confusion_matrix(y_test, logistic_predictions, labels=labels)

ConfusionMatrixDisplay(
    confusion_matrix=cm_logistic,
    display_labels=labels
).plot(cmap="Blues")

plt.title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.show()

### How to read the confusion matrix
- Rows represent the actual rating.
- Columns represent the model's predicted rating.
- Values on the diagonal are correct predictions.
- Values outside the diagonal are incorrect predictions.

Look carefully at rating 2 because it is the smaller class in this dataset. A model can have good overall accuracy while missing many employees in a smaller class.

## 10. Build Model 2: Random Forest

Random Forest combines multiple decision trees. Each tree makes predictions, and the forest combines their results. It can learn more complex patterns than a simple linear model.

We use the same preprocessing pipeline and the same training/test split for a fair comparison.

In [ ]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

random_forest_model.fit(X_train, y_train)

print("Random Forest model training completed.")

## 11. Evaluate Model 2

In [ ]:
rf_predictions = random_forest_model.predict(X_test)

print("Random Forest Accuracy:",
      round(accuracy_score(y_test, rf_predictions), 4))

print("\\nClassification Report:")
print(classification_report(y_test, rf_predictions, zero_division=0))

In [ ]:
cm_rf = confusion_matrix(y_test, rf_predictions, labels=labels)

ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=labels
).plot(cmap="Greens")

plt.title("Random Forest — Confusion Matrix")
plt.tight_layout()
plt.show()

## 12. Compare the models

Accuracy measures the fraction of all test predictions that were correct. Weighted F1-score combines precision and recall while accounting for the number of examples in each class.

Because rating 3 is much more common than ratings 2 and 4, do not choose a model using accuracy alone. Compare the classification reports and pay attention to recall and F1-score for each rating.

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, logistic_predictions),
        accuracy_score(y_test, rf_predictions)
    ],
    "Weighted_F1": [
        f1_score(y_test, logistic_predictions, average="weighted"),
        f1_score(y_test, rf_predictions, average="weighted")
    ],
    "Macro_F1": [
        f1_score(y_test, logistic_predictions, average="macro"),
        f1_score(y_test, rf_predictions, average="macro")
    ]
})

model_comparison.round(4)

In [ ]:
model_comparison.set_index("Model")[["Accuracy", "Weighted_F1", "Macro_F1"]].plot(
    kind="bar", figsize=(9,5)
)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 13. Check Random Forest feature importance

Random Forest can estimate how much each transformed input feature contributes to its tree decisions. Because categorical columns are one-hot encoded, their category levels appear as separate features.

**Caution:** feature importance is a model-based measure of predictive usefulness, not proof that a feature causes performance ratings. Correlated features can share or distort importance.

In [ ]:
rf_preprocessor = random_forest_model.named_steps["preprocessor"]
rf_classifier = random_forest_model.named_steps["classifier"]

feature_names = rf_preprocessor.get_feature_names_out()
importance_values = rf_classifier.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
}).sort_values("Importance", ascending=False)

feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(10,7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 14. Predict a rating for one employee record (example)

This example uses one row from the held-out test data. It shows how the trained model makes a prediction. The row is selected from the dataset, so it is only a demonstration, not a new employee assessment.

In [ ]:
sample_employee = X_test.iloc[[0]]

predicted_rating = random_forest_model.predict(sample_employee)[0]

print("Predicted performance rating:", predicted_rating)
print("Actual performance rating:", y_test.iloc[0])

## 15. Save the trained model (optional)

This saves the complete Random Forest pipeline, including preprocessing and the trained classifier. Install `joblib` if needed. Keep the saved model file with the project files if you want to reuse it later.

In [ ]:
import joblib

joblib.dump(random_forest_model, "employee_performance_random_forest.pkl")
print("Model saved as employee_performance_random_forest.pkl")

## 16. Model-building conclusion — complete after running

Use your actual outputs to report:

1. Which model achieved the better test accuracy?
2. Which model achieved the better macro F1-score?
3. How well did each model predict rating 2?
4. How well did each model predict rating 3?
5. How well did each model predict rating 4?
6. Which features had the highest Random Forest importance?
7. What limitations should be considered?

### Important project limitations
- The rating classes are imbalanced, with rating 3 being much more common.
- Model results depend on the train/test split and the selected algorithms.
- Feature importance indicates predictive patterns, not causes.
- Some variables may be related to one another.
- Salary hike and promotion variables may reflect previous performance or company decisions. Their presence does not establish the direction of influence.
- Do not claim that the model proves why an employee performs poorly. It predicts ratings based on patterns in the available data.

# Part 4 — Overall Findings and Recommendations

After running the notebook, use the actual tables, charts, and model metrics generated above to complete the observations below.

## 1. Key findings to report
- Overall performance-rating distribution:
- Department with the highest average performance rating:
- Department with the lowest average performance rating:
- Three factors worth investigating based on the EDA and model outputs:
- Best-performing model based on appropriate evaluation metrics:

## 2. Recommendations for INX Future Inc.
1. Review employee work-environment feedback and identify practical improvements.
2. Investigate departments with lower average performance ratings, considering team size and job-role differences.
3. Review career development, promotion pathways, and access to relevant training.
4. Use multiple sources of evidence in performance reviews rather than relying on a single rating or model prediction.
5. Validate model performance and fairness before considering any operational use.

## 3. Project limitations
- The analysis is based on one dataset containing 1,200 employee records.
- Performance ratings are imbalanced; rating 3 is the majority class.
- Descriptive comparisons do not establish cause and effect.
- Small department or subgroup counts can make percentages unstable.
- The dataset may not contain every factor that influences employee performance.
- Model results must be interpreted using class-level recall, macro F1-score, and confusion matrices—not accuracy alone.
- A prediction model should support human review, not replace it.

## 4. Final conclusion

Write a short conclusion using the actual outputs after running all cells. Summarize the department comparison, the strongest patterns observed, the model comparison, and the recommendations. Do not claim a model accuracy or a top-three feature unless it is supported by the notebook output.
